<a href="https://colab.research.google.com/github/zshaan25/CodeAlpha_Disease_Prediction/blob/main/Disease_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score
import xgboost as xgb

# 1. Fetch Medical Data
print("Fetching Breast Cancer Wisconsin Diagnostic Dataset...")
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target # 0: malignant, 1: benign

# 2. Train-Test Split (Using stratify to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Feature Scaling
# Crucial step: SVM and Logistic Regression are highly sensitive to unscaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Define the Base Learners
# We select algorithms with different mathematical approaches to ensure diverse decision boundaries
log_clf = LogisticRegression(random_state=42, max_iter=1000)
svm_clf = SVC(probability=True, random_state=42) # probability=True is strictly required for 'soft' voting
xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# 5. Construct the Voting Classifier
# We use 'soft' voting to compute the convex combination of predicted probabilities from the base learners
voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('svc', svm_clf), ('xgb', xgb_clf)],
    voting='soft'
)

# 6. Execute Training
print("\nTraining the Voting Classifier Ensemble...")
voting_clf.fit(X_train_scaled, y_train)

# 7. Evaluate Medical Performance
# In disease prediction, maximizing Recall (minimizing false negatives) is the primary engineering objective.
y_pred = voting_clf.predict(X_test_scaled)

print("\n--- Final Model Evaluation ---")
print(f"Ensemble Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Ensemble Recall:    {recall_score(y_test, y_pred):.4f}\n")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Fetching Breast Cancer Wisconsin Diagnostic Dataset...

Training the Voting Classifier Ensemble...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:11:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Final Model Evaluation ---
Ensemble Precision: 0.9726
Ensemble Recall:    0.9861

Detailed Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.95      0.96        42
      benign       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

